In [ ]:
import h3
pontos = [(-23.460322, -46.570984), (-23.459669, -46.571031), (-23.460373, -46.571893)]
for res in [8, 9, 10, 11]:
    celulas = {h3.latlng_to_cell(lat, lon, res) for lat, lon in pontos}
    print(res, f"{h3.average_hexagon_edge_length(res, unit='m'):.0f} m de aresta:", len(celulas), "célula(s)")

In [ ]:
import sys
sys.path.insert(0, "..")

import h3
import numpy as np
import pandas as pd

from src.comum import normalizar
from src.silver import ler_silver

RESOLUCAO = 9  # aresta média ~200 m
chamados, _, manifesto = ler_silver()
chamados["celula"] = [h3.latlng_to_cell(lat, lon, RESOLUCAO) for lat, lon in zip(chamados["lat"], chamados["lon"])]
print(f"{len(chamados):,} chamados em {chamados['celula'].nunique():,} células")


In [ ]:
def distancia_m(lat1, lon1, lat2, lon2):
    """Haversine vetorizado, em metros."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(a))


def pares_mesma_rua(df, k=1):
    """Todos os pares (chamado, vizinho) na mesma rua e no disco de células de raio k.

    Com resolução 9 e k=1, o disco cobre com certeza qualquer ponto a ~300 m: suficiente
    para separar as faixas até "200 m ou mais".
    """
    base = df.loc[df["logradouro"].notna(), ["lat", "lon", "data_abertura", "celula", "logradouro"]].copy()
    base["rua"] = base["logradouro"].map({v: normalizar(v) for v in base["logradouro"].unique()})
    base["id"] = base.index
    busca = base.assign(celula_busca=base["celula"].map(lambda c: list(h3.grid_disk(c, k)))).explode("celula_busca")
    pares = busca.merge(base, left_on=["celula_busca", "rua"], right_on=["celula", "rua"], suffixes=("", "_viz"))
    pares = pares[pares["id"] != pares["id_viz"]].copy()
    pares["dias"] = (pares["data_abertura"] - pares["data_abertura_viz"]).abs().dt.days
    pares["dist"] = distancia_m(pares["lat"], pares["lon"], pares["lat_viz"], pares["lon_viz"])
    return pares[["id", "id_viz", "dias", "dist"]], len(base)


def mais_proximo(pares, dias_min, dias_max):
    """Distância até o vizinho mais próximo aberto entre dias_min e dias_max de diferença."""
    janela = pares[pares["dias"].between(dias_min, dias_max)]
    return janela.groupby("id")["dist"].min()


pares, n_com_rua = pares_mesma_rua(chamados)
print(f"{len(pares):,} pares candidatos (mesma rua, células vizinhas)")


In [ ]:
FAIXAS = [0, 25, 50, 100, 200, np.inf]
ROTULOS = ["0–25 m", "25–50 m", "50–100 m", "100–200 m", "200 m ou mais"]

proximo_7d = mais_proximo(pares, 0, 7)
faixa = pd.cut(proximo_7d, FAIXAS, right=False, labels=ROTULOS)
resumo = faixa.value_counts().reindex(ROTULOS).to_frame("chamados")
resumo["largura (m)"] = [25, 25, 50, 100, np.nan]
resumo["chamados por metro"] = (resumo["chamados"] / resumo["largura (m)"]).round(0)
print(resumo.to_string())
sem_vizinho = n_com_rua - len(proximo_7d)
print(f"\nSem vizinho na mesma rua em ±7 dias (até ~300 m): {sem_vizinho:,} ({sem_vizinho / n_com_rua:.1%})")


In [ ]:
# A distância ao vizinho MAIS PRÓXIMO sempre decresce, mesmo para pontos ao acaso; por isso o vale
# pode não aparecer. Controle: a mesma busca com chamados abertos meses depois (mesma rua, quase
# certamente outro buraco). O limite é onde a janela de 7 dias deixa de ter excesso sobre o controle.
faixas_20m = np.arange(0, 320, 20)
contar = lambda serie: pd.cut(serie, faixas_20m, right=False).value_counts().sort_index()
comparacao = pd.DataFrame({
    "±7 dias": contar(proximo_7d),
    "controle 90–97 dias": contar(mais_proximo(pares, 90, 97)),
    "controle 180–187 dias": contar(mais_proximo(pares, 180, 187)),
})
comparacao["razão (90d)"] = (comparacao["±7 dias"] / comparacao["controle 90–97 dias"]).round(2)
comparacao["razão (180d)"] = (comparacao["±7 dias"] / comparacao["controle 180–187 dias"]).round(2)
print(comparacao.to_string())


In [ ]:
def votar(alvos, eleitores, k=0):
    """Distrito mais comum entre os eleitores nas células do disco de raio k de cada alvo."""
    votos = eleitores.groupby(["celula", "distrito"]).size().rename("n").reset_index()
    celulas = alvos["celula"].map(lambda c: list(h3.grid_disk(c, k))).explode().rename("celula")
    apuracao = (celulas.rename_axis("alvo").reset_index()
                .merge(votos, on="celula")
                .groupby(["alvo", "distrito"])["n"].sum().reset_index())
    apuracao["votos"] = apuracao.groupby("alvo")["n"].transform("sum")
    apuracao["proporcao"] = apuracao["n"] / apuracao["votos"]
    vencedor = apuracao.sort_values("n", ascending=False).drop_duplicates("alvo").set_index("alvo")
    return vencedor[["distrito", "proporcao", "votos"]].reindex(alvos.index)


def votar_com_ampliacao(alvos, eleitores):
    """Vota na própria célula; quem ficou sem voto tenta de novo com as vizinhas (k=1)."""
    resultado = votar(alvos, eleitores, k=0).assign(nivel="célula")
    faltam = resultado["distrito"].isna()
    resultado.loc[faltam] = votar(alvos[faltam], eleitores, k=1).assign(nivel="k=1")
    resultado.loc[resultado["distrito"].isna(), "nivel"] = "sem votos"
    return resultado


com_distrito = chamados[chamados["distrito"].notna()]


In [ ]:
# A amostra sai dos eleitores: senão cada chamado votaria em si mesmo e o acerto seria inflado.
amostra = com_distrito.sample(5000, random_state=42)
previsto = votar_com_ampliacao(amostra, com_distrito.drop(amostra.index))
acertou = previsto["distrito"].eq(amostra["distrito"]).fillna(False)

print(f"Acerto geral: {acertou.mean():.1%} (sem resultado: {previsto['distrito'].isna().sum()})")
faixa_votos = pd.cut(previsto["proporcao"], [0, 0.6, 0.8, 1.0001], right=False, labels=["< 60%", "60–80%", "≥ 80%"])
validacao = pd.DataFrame({"chamados": faixa_votos.value_counts(),
                          "acerto": acertou.groupby(faixa_votos, observed=False).mean().round(3)})
print(validacao.sort_index().to_string())


In [ ]:
sem_distrito = chamados[chamados["distrito"].isna()]
atribuido = votar_com_ampliacao(sem_distrito, com_distrito)
atribuido["divisa"] = atribuido["proporcao"] < 0.8   # abaixo de 80% o acerto cai para ~60%
print(atribuido["nivel"].value_counts().to_string())
print(f"\nDivisa (vencedor com < 80% dos votos): {int(atribuido['divisa'].sum())}")
print("Sem votos nem nas vizinhas (provavelmente fora do município):")
print(sem_distrito.loc[atribuido["nivel"] == "sem votos", ["lat", "lon", "logradouro"]].round(4).head(10).to_string())
